<a href="https://colab.research.google.com/github/FerFilho23/Deep-Learning-Soccer-Tracker/blob/main/Soccer_Tracker_YOLO26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Requirements (Ultralytics)

In [2]:
!pip install ultralytics


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# Train Baseline Model

**1-Model architecture & size (`model`):**

There are several YOLO26 models sizes available to train, including `yolo26n-obb.pt` `yolo26s-obb.pt` `yolo26m-obb.pt` `yolo26l-obb.pt` `yolo26x-obb.pt` . Larger models run slower but have higher accuracy, while smaller models run faster but have lower accuracy.
If you aren't sure which model size to use, `yolo26s.pt` is a good starting point.



<br/>

**2-Number of epochs (`epochs`)**

In machine learning, one “epoch” is one single pass through the full training
| | Goal | Dataset Size | Recommended Epochs | Notes |
|:---:|------|--------------|-------------------|-------|
| 🧪 | Quick Prototype | Any | 20–30 | Just testing if pipeline works |
| 🟡 | Low Accuracy | < 200 images | 40–60 | Baseline usable model |
| 🟠 | Mid Accuracy | 200–500 images | 60–100 | Good balance of speed vs. quality |
| 🟢 | High Accuracy | 500–2000 images | 100–150 | Use best.pt, monitor mAP |
| 🔵 | Production Quality | 2000+ images | 150–300 | Diminishing returns after ~200 |
| ⚠️ | Overfitting Risk Zone | Any | 300+ | Only if val loss is still improving |

<br/>

3- **Resolution guid (`imgsz`)**

Resolution has a large impact on the speed and accuracy of the model: a lower resolution model will have higher speed but less accuracy.

| Resolution | Use Case | Object Size | Speed | Accuracy | VRAM Usage |
|:----------:|----------|:-----------:|:-----:|:--------:|:----------:|
| **320×320** | Edge devices, real-time IoT | Large objects only | ⚡⚡⚡⚡ Very Fast | ⭐ Low | 🟢 Very Low |
| **480×480** | Mobile apps, low-power GPUs | Medium–Large | ⚡⚡⚡ Fast | ⭐⭐ Medium-Low | 🟢 Low |
| **640×640** | ✅ Default / General purpose | Medium objects | ⚡⚡ Moderate | ⭐⭐⭐ Good | 🟡 Moderate |
| **800×800** | Detailed scenes, mid-size objects | Small–Medium | ⚡ Slower | ⭐⭐⭐⭐ High | 🟠 High |
| **1280×1280** | Drone footage, sports balls, screws | Small objects | 🐢 Slow | ⭐⭐⭐⭐⭐ Very High | 🔴 High |
| **1920×1920** | Microscopy, PCB inspection, tiny defects | Extremely small | 🐢🐢 Very Slow | ⭐⭐⭐⭐⭐ Max | 🔴🔴 Extreme |

<br/>

**Real-World Scenario Guide**

| Scenario | Recommended Resolution | Why |
|----------|----------------------|-----|
| 🎾 Sports ball tracking | **1280×1280** | Balls are tiny in wide-angle footage |
| 🔩 Defect / screw detection | **1280–1920** | Sub-centimeter details matter |
| 🚗 Car / vehicle detection | **640×640** | Cars are large, default works great |
| 🚶 Person detection (crowd) | **800×800** | People can be small in wide shots |
| 🍬 Candy / product counting | **640×640** | Objects fill reasonable frame space |
| 🛸 Drone aerial surveillance | **1280×1280** | Camera is far, objects appear tiny |
| 🏭 Assembly line inspection | **800–1280** | Depends on part size |
| 📱 Phone/webcam real-time | **480×480** | Prioritize speed for live feed |
| 🍓 Fruit sorting (close-up) | **640×640** | Objects are close and large enough |
| 🔬 Medical / microscopy | **1280–1920** | Tiny structures need max detail |


<br/> <br/>

In [6]:
!yolo detect train \
    data=../dataset/processed/data.yaml \
    model=yolo26n.pt \
    epochs=80 \
    imgsz=960 \
    batch=4 \
    device=mps \
    workers=2 \
    patience=20 \
    name=yolo26n_baseline_960

New https://pypi.org/project/ultralytics/8.4.103 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.96 🚀 Python-3.13.5 torch-2.13.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../dataset/processed/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

# Test the Model

!yolo detect predict model=../runs/detect/yolo26n_baseline_960/weights/best.pt source=../dataset/processed/valid/images save=True